# Talanta GPU render test (internal, private)

Renders the 24-frame L07 orbit (scroll_test_01) on the Kaggle GPU, to compare with 704 s a frame on the 4-core cloud CPU.
Settings: GPU T4 x2, Internet on. Input: the private dataset `talanta-render-pack` (the cloud's render pack).
People, cars and lights in the frames are Visualization. The frames are internal only. Do not publish this notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1; free -g | head -2

In [ ]:
import glob, os, subprocess, tarfile
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"  # same version as the cloud bpy 5.2.2
if not os.path.exists("/kaggle/working/blender"):
    subprocess.run(f"wget -q {BLENDER_URL} -O /tmp/b.tar.xz && mkdir -p /kaggle/working/blender && "
                   "tar -xf /tmp/b.tar.xz -C /kaggle/working/blender --strip-components=1 && rm /tmp/b.tar.xz", shell=True, check=True)
BLENDER = "/kaggle/working/blender/blender"
subprocess.run([BLENDER, "-b", "--version"], check=True)

In [ ]:
# Unpack the render pack into /kaggle/working/pack/talanta_render_pack (the pack root holds blender/ and blender/out/frozen_FK.blend).
# Kaggle can keep the upload as a .tar or unpack it; both cases work. The input is read-only, so copy it to a writable place.
import shutil
PACK = "/kaggle/working/pack/talanta_render_pack"
if not os.path.exists(PACK):
    os.makedirs("/kaggle/working/pack", exist_ok=True)
    unpacked = glob.glob("/kaggle/input/**/talanta_render_pack/blender/out/frozen_FK.blend", recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit("/blender/", 1)[0], PACK)
    else:
        parts = sorted(glob.glob("/kaggle/input/**/talanta_render_pack.tar*", recursive=True))
        parts = [q for q in parts if not q.endswith(".sha256")]
        print(parts)
        subprocess.run(f"cat {' '.join(parts)} | tar -x -C /kaggle/working/pack", shell=True, check=True)
print(os.listdir(PACK))

# The pack has no blender/kaggle/gpu_pre.py, so write it here (same as blender/kaggle/gpu_pre.py in the repo).
GPU_PRE = "/kaggle/working/gpu_pre.py"
open(GPU_PRE, "w").write('''prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for kind in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = kind
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == kind for d in prefs.devices):
        chosen = kind
        break
if chosen is None:
    raise SystemExit("gpu_pre.py: no OptiX or CUDA device found")
for d in prefs.devices:
    d.use = d.type == chosen
s.cycles.device = "GPU"
print("gpu_pre.py:", chosen, [d.name for d in prefs.devices if d.use])
''')


In [ ]:
def render(out, *args):
    env = dict(os.environ, TALANTA_SITE_BLEND=f"{PACK}/blender/out/frozen_FK.blend", TALANTA_PRE=GPU_PRE)
    cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1",
           "--python", f"{PACK}/blender/scroll_test/render_scroll_frames.py", "--", out, *args]
    subprocess.run(cmd, cwd=PACK, env=env, check=True)

# Step 1: one frame at the scroll_test_01 settings (128 samples, haze on). Check the GPU line and the time.
render("blender/out/kaggle_t1", "frames=24", "only=0-0")

In [ ]:
# Step 2: the full 24 frames at the scroll_test_01 settings.
render("blender/out/kaggle_24", "frames=24")

In [ ]:
# Step 3: the cheap levers on 5 frames: 48 samples, haze volume off.
render("blender/out/kaggle_levers", "frames=24", "only=0-4", "samples=48", "haze=0")

In [ ]:
import json
for d in ("kaggle_t1", "kaggle_24", "kaggle_levers"):
    for f in glob.glob(f"{PACK}/blender/out/{d}/*.json"):
        print(d, f); print(open(f).read()[:3000])
# Keep the logs and a contact sheet only. Do not save the frames as notebook output.
subprocess.run(f"mkdir -p /kaggle/working/logs && cp {PACK}/blender/out/kaggle_*/*.json /kaggle/working/logs/", shell=True)